# Games for today and tomorrow

Loads your pipeline **games** Parquet and filters rows where `game_date` is today or tomorrow (local timezone).

**Default file:** `data/games_{season}.parquet` where `season` is **`PIPELINE_SEASON`** (if set) or the **calendar year**. Run **`python -m app.main live`** (or Docker **`compose --profile live up live`**) to refresh the current season; use **`backfill --years ...`** for one-off seasons.

Legacy fallbacks: `games.parquet`, `games_live.parquet` (only if present).

**Requires** a schedule window that actually includes *today's* calendar date. If the notebook prints a `game_date` range ending in 2024 but your clock is 2026, the "today / tomorrow" filter will correctly show **0 rows** until you rebuild for the current season.

Run Jupyter with working directory `mlb-model` **or** `mlb-model/notebooks` (the next cell resolves `data/` automatically).

The load cell sets **`display.max_columns` to 50** (raise to **`None`** in that cell to show every column with no cap). The main table shows **all** columns in `sub` for today/tomorrow. A later section flags **mismatch / favorite** games using `MismatchBreakdown`-style thresholds plus optional platoon and bullpen filters.


In [6]:
import os
from datetime import datetime
from pathlib import Path

import pandas as pd

# Show many columns in DataFrame HTML / repr (raise to None for no limit).
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 50)

_cwd = Path.cwd()
DATA = _cwd / "data" if (_cwd / "data").is_dir() else _cwd.parent / "data"

_ps = os.environ.get("PIPELINE_SEASON", "").strip()
_season = int(_ps) if _ps.isdigit() else datetime.now().year

# Prefer games_{season}.parquet; then generic / legacy names. If several exist, use newest mtime.
_candidates = [
    DATA / f"games_{_season}.parquet",
    DATA / "games.parquet",
    DATA / "games_live.parquet",
]
_existing = [p for p in _candidates if p.is_file()]
if not _existing:
    raise FileNotFoundError(
        f"No games parquet in {DATA} (tried games_{_season}.parquet, games.parquet, games_live.parquet)"
    )
PARQUET = max(_existing, key=lambda p: p.stat().st_mtime)
if len(_existing) > 1:
    print("Multiple game parquets found; using newest by file time:", PARQUET.name)

print("Using:", PARQUET.resolve())
df = pd.read_parquet(PARQUET)
df["game_date"] = pd.to_datetime(df["game_date"]).dt.normalize()
gd_min, gd_max = df["game_date"].min(), df["game_date"].max()
print(f"game_date range: {gd_min.date()} .. {gd_max.date()}  ({len(df)} rows)")
if "stats_season" in df.columns:
    ss = df["stats_season"].dropna().unique()
    print("stats_season values:", sorted(ss.tolist()))
clock = pd.Timestamp.today().normalize()
if clock < gd_min or clock > gd_max:
    print(
        f"\n>>> Today's date ({clock.date()}) is outside this file's game_date range.\n"
        "    Re-run the pipeline for the season you care about, e.g. (host):\n"
        "    python -m app.main live --season 2026\n"
        "    Or: docker compose --profile live up live  (set PIPELINE_SEASON if needed)"
    )

Using: /Users/austinlolli/Code/baseball/mlb-model/data/games_2026.parquet
game_date range: 2026-03-01 .. 2026-09-27  (2796 rows)
stats_season values: [2026]


In [7]:
# df already loaded and game_date normalized in previous cell
today = pd.Timestamp.today().normalize()
tomorrow = today + pd.Timedelta(days=1)
# sub = df[df["game_date"].isin([today, tomorrow])].copy()
sub = df[df["game_date"].isin([today])].copy()

print(f"Today={today.date()}, tomorrow={tomorrow.date()}, rows={len(sub)}")

Today=2026-04-16, tomorrow=2026-04-17, rows=10


In [8]:
# All columns for today/tomorrow (respects display.max_columns above).
view = sub.sort_values(["game_date", "home_team_name"])

_datapoints = [
    "game_date",
    "home_team_name",
    "away_team_name",
    # batting stats
    "home_wrc_plus",
    "away_wrc_plus",
    "offense_diff",
    "home_offense_platoon",
    "away_offense_platoon",
    "offense_platoon_diff",
    # starting pitcher stats
    "home_probable_pitcher",
    "home_sp_throws",
    "home_sp_kbb",
    "home_sp_kbb_roll14",
    "home_sp_xfip",
    "home_sp_xfip_roll14",
    "away_probable_pitcher",
    "away_sp_throws",
    "away_sp_kbb",
    "away_sp_kbb_roll14",
    "away_sp_xfip",
    "away_sp_xfip_roll14"
    "sp_kbb_diff",
    "sp_xfip_diff",
    #bullpen stats
    "home_pen_season_era",
    "home_pen_roll14_era",
    "home_pen_season_fip",
    "home_pen_roll14_fip",
    "home_pen_roll14_minus_season_fip",
    "away_pen_season_era",
    "away_pen_roll14_era",
    "away_pen_season_fip",
    "away_pen_roll14_fip",
    "away_pen_roll14_minus_season_fip",	
    # kalshi
    "kalshi_home_implied",
    "kalshi_away_implied",	
    "edge_vs_model"
]
    

data = [c for c in _datapoints if c in view.columns]
ids = [c for c in view.columns if c not in _datapoints]

organized_view = view[data + ids]
organized_view


,game_date,home_team_name,away_team_name,home_wrc_plus,away_wrc_plus,offense_diff,home_offense_platoon,away_offense_platoon,offense_platoon_diff,home_probable_pitcher,home_sp_throws,home_sp_kbb,home_sp_kbb_roll14,home_sp_xfip,home_sp_xfip_roll14,away_probable_pitcher,away_sp_throws,away_sp_kbb,away_sp_kbb_roll14,away_sp_xfip,sp_xfip_diff,home_pen_season_era,home_pen_roll14_era,home_pen_season_fip,home_pen_roll14_fip,home_pen_roll14_minus_season_fip,away_pen_season_era,away_pen_roll14_era,away_pen_season_fip,away_pen_roll14_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,game_pk,detailed_state,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,sp_kbb_diff,away_sp_xfip_roll14,stats_season
645,2026-04-16,Athletics,Texas Rangers,94.856601,97.700877,-2.844276,99.704683,74.003517,25.701166,Jacob Lopez,L,-1.111111,6.521739,6.263636,7.492857,Jack Leiter,R,16.666667,20.512821,4.132787,2.130849,4.263158,4.928571,4.047368,4.219048,0.171679,2.741117,3.266129,4.105076,4.406452,0.301375,0.145,0.855,NaN,825023,In Progress,133,140,3.0,5.0,682052,683004,NaN,True,2026,2026,-17.777778,4.830769,2026
644,2026-04-16,Chicago White Sox,Tampa Bay Rays,85.612704,104.527139,-18.914435,98.475967,106.032907,-7.556940,Jordan Leasure,R,12.500000,13.636364,5.035484,4.864706,Steven Matz,L,17.647059,26.190476,3.334375,1.701109,5.618497,3.342857,4.521965,3.471429,-1.050537,6.982759,5.869565,5.737931,5.369565,-0.368366,0.005,0.995,NaN,824614,Final,145,139,3.0,5.0,673929,571927,0.0,True,2026,2026,-5.147059,1.736364,2026
640,2026-04-16,Cincinnati Reds,San Francisco Giants,89.879118,90.021332,-0.142214,87.188862,83.532555,3.656307,Chase Burns,R,14.772727,11.764706,4.085075,5.923529,Landen Roupp,R,18.888889,16.666667,2.173529,1.911545,2.712919,2.045455,4.176555,4.100000,-0.076555,4.165714,4.621622,4.334286,3.991892,-0.342394,0.005,0.995,NaN,824530,Final,113,137,0.0,3.0,695505,694738,0.0,True,2026,2026,-4.116162,1.881250,2026
646,2026-04-16,Cleveland Guardians,Baltimore Orioles,99.123015,103.389429,-4.266414,97.173393,116.647128,-19.473734,Parker Messick,L,16.666667,13.043478,2.873585,3.614286,Shane Baz,R,11.267606,8.000000,3.600000,-0.726415,5.000000,5.625000,4.163492,4.300000,0.136508,3.320856,2.288136,3.404813,3.201695,-0.203118,0.545,0.455,NaN,824453,Warmup,114,110,0.0,0.0,800048,669358,NaN,True,2026,2026,5.399061,4.318750,2026
641,2026-04-16,Detroit Tigers,Kansas City Royals,100.687367,92.012325,8.675041,89.976553,93.938968,-3.962414,Keider Montero,R,20.634921,21.052632,1.630612,1.745161,Kris Bubic,L,17.977528,32.608696,3.276471,-1.645858,2.945455,1.657895,4.227273,3.652632,-0.574641,4.291391,4.736842,4.868212,5.100000,0.231788,0.305,0.695,NaN,824289,In Progress,116,118,7.0,8.0,672456,663460,NaN,True,2026,2026,2.657393,2.016667,2026
647,2026-04-16,Houston Astros,Colorado Rockies,113.628822,96.847594,16.781228,111.517367,95.345240,16.172128,Ryan Weiss,R,16.071429,13.953488,5.554545,5.350000,Juan Mejia,R,2.702703,4.545455,4.900000,0.654545,6.068807,6.889655,6.044954,6.141379,0.096425,3.753363,3.331169,4.337668,4.268831,-0.068837,0.605,0.395,NaN,824208,Pre-Game,117,115,0.0,0.0,680802,675848,NaN,True,2026,2026,13.368726,5.900000,2026
643,2026-04-16,Milwaukee Brewers,Toronto Blue Jays,102.251718,96.847594,5.404124,87.631887,95.204613,-7.572725,Brandon Sproat,R,6.756757,3.125000,6.217647,5.418182,Patrick Corbin,L,17.073171,9.523810,5.168966,1.048682,4.153846,6.428571,3.248352,3.785714,0.537363,4.063107,3.425373,3.376699,3.502985,0.126286,0.995,0.005,NaN,823801,Final,158,141,2.0,1.0,687075,571578,1.0,True,2026,2026,-10.316414,10.350000,2026
642,2026-04-16,New York Yankees,Los Angeles Angels,98.838587,105.095994,-6.257407,78.399766,112.690504,-34.290739,Max Fried,L,11.811024,12.280702,3.010000,2.350000,Brent Suter,L,15.254237,11.764706,4.033333,-1.023333,3.469274,4.275000,2.831844,3.250000,0.418156,4.613065

## Pitch / offense mismatches ("favorites")

Base criteria from `MismatchBreakdown.ipynb` (SP K-BB edge, SP xFIP edge, offense edge), but this notebook now supports **N-of-3 matching**:

- Example default: **at least 2 of 3** core criteria for home-favored or away-favored.
- Optional extras can still be layered on:
  - **Platoon:** `offense_platoon_diff` agrees with favored side.
  - **Bullpen:** favored side has lower `*_pen_roll14_fip`.

The output also includes a simple **`mismatch_score`** so you can rank candidates while we refine a fuller scoring model.

In [9]:
# --- Core thresholds from MismatchBreakdown.ipynb ---
SP_KBB_ABS = 3.0
SP_XFIP_ABS = 0.5
OFFENSE_ABS = 10.0
MIN_CORE_MATCHES = 2  # out of 3 core checks (2 = "most", 3 = strict old behavior)

# --- Optional: platoon + bullpen roll14 (lower FIP = better pen) ---
USE_PLATOON_EXTRA = True
PLATOON_MIN = 0.0  # require |offense_platoon_diff| > this when extras on
USE_PEN_EXTRA = True
PEN_ROLL_MIN_GAP = 0.0  # favored pen FIP must be lower by this much vs opponent pen

s = sub.copy()
req = {"sp_kbb_diff", "sp_xfip_diff", "offense_diff"}
missing = req - set(s.columns)
if missing:
    raise ValueError(f"Missing columns for mismatch filter: {missing}")

# Core directional checks for each side
home_core_kbb = s["sp_kbb_diff"] > SP_KBB_ABS
home_core_xfip = s["sp_xfip_diff"] < -SP_XFIP_ABS
home_core_off = s["offense_diff"] > OFFENSE_ABS
home_core_n = home_core_kbb.astype(int) + home_core_xfip.astype(int) + home_core_off.astype(int)

away_core_kbb = s["sp_kbb_diff"] < -SP_KBB_ABS
away_core_xfip = s["sp_xfip_diff"] > SP_XFIP_ABS
away_core_off = s["offense_diff"] < -OFFENSE_ABS
away_core_n = away_core_kbb.astype(int) + away_core_xfip.astype(int) + away_core_off.astype(int)

home_base = home_core_n >= MIN_CORE_MATCHES
away_base = away_core_n >= MIN_CORE_MATCHES

home_ext = home_base.copy()
away_ext = away_base.copy()

if USE_PLATOON_EXTRA and "offense_platoon_diff" in s.columns:
    po = s["offense_platoon_diff"].fillna(0)
    home_ext = home_ext & (po > PLATOON_MIN)
    away_ext = away_ext & (po < -PLATOON_MIN)

if USE_PEN_EXTRA and {"home_pen_roll14_fip", "away_pen_roll14_fip"}.issubset(s.columns):
    hr = s["home_pen_roll14_fip"]
    ar = s["away_pen_roll14_fip"]
    g = PEN_ROLL_MIN_GAP
    # Home favored: home pen better (lower FIP) than away
    pen_ok_home = (hr + g < ar) | hr.isna() | ar.isna()
    # Away favored: away pen better than home
    pen_ok_away = (ar + g < hr) | hr.isna() | ar.isna()
    home_ext = home_ext & pen_ok_home
    away_ext = away_ext & pen_ok_away

# Keep ALL rows so you can inspect near-misses, and add match flags.
# (Old strict filtering is intentionally disabled.)
favorites = s.copy().sort_values(["game_date", "home_team_name"])

favorites["home_core_matches"] = home_core_n.astype(int)
favorites["away_core_matches"] = away_core_n.astype(int)
favorites["home_base_match"] = home_base
favorites["away_base_match"] = away_base
favorites["home_with_extras"] = home_ext
favorites["away_with_extras"] = away_ext

# Preferred side for scoring / display = stronger core count (tie -> absolute edge blend).
_tie_home = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + (-favorites["sp_xfip_diff"]).abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
_tie_away = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + favorites["sp_xfip_diff"].abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
prefer_home = (favorites["home_core_matches"] > favorites["away_core_matches"]) | (
    (favorites["home_core_matches"] == favorites["away_core_matches"]) & (_tie_home >= _tie_away)
)

favorites["_mismatch_side"] = prefer_home.map({True: "home_favored", False: "away_favored"})
favorites["core_matches"] = favorites[["home_core_matches", "away_core_matches"]].max(axis=1)
favorites["favored_team"] = favorites.apply(
    lambda r: r["home_team_name"] if r["_mismatch_side"] == "home_favored" else r["away_team_name"],
    axis=1,
)

# Basic mismatch score (higher = stronger favorite setup)
favorites["mismatch_score"] = (
    (favorites["sp_kbb_diff"].abs() / SP_KBB_ABS)
    + (favorites["sp_xfip_diff"].abs() / SP_XFIP_ABS)
    + (favorites["offense_diff"].abs() / OFFENSE_ABS)
)
if "offense_platoon_diff" in favorites.columns:
    favorites["mismatch_score"] += 0.35 * (favorites["offense_platoon_diff"].abs() / 10.0)
if {"home_pen_roll14_fip", "away_pen_roll14_fip"}.issubset(favorites.columns):
    pen_gap = (favorites["away_pen_roll14_fip"] - favorites["home_pen_roll14_fip"]).where(
        favorites["_mismatch_side"] == "home_favored",
        favorites["home_pen_roll14_fip"] - favorites["away_pen_roll14_fip"],
    )
    favorites["mismatch_score"] += 0.35 * (pen_gap.fillna(0) / 0.75)

favorites = favorites.sort_values(["mismatch_score", "core_matches", "game_date"], ascending=[False, False, True])

_prio = [
    "game_date",
    "detailed_state",
    "favored_team",
    "_mismatch_side",
    "core_matches",
    "mismatch_score",
    "home_team_name",
    "away_team_name",
    "sp_kbb_diff",
    "sp_xfip_diff",
    "offense_diff",
    "offense_platoon_diff",
    "home_offense_platoon",
    "away_offense_platoon",
    "home_pen_roll14_fip",
    "away_pen_roll14_fip",
    "home_pen_season_fip",
    "away_pen_season_fip",
    "home_probable_pitcher",
    "away_probable_pitcher",
]
_prio = [c for c in _prio if c in favorites.columns]
_rest = [c for c in favorites.columns if c not in _prio]
favorites_view = favorites[_prio + _rest]

print(
    f"Core filter (>= {MIN_CORE_MATCHES}/3): home={int(home_base.sum())}, away={int(away_base.sum())} | "
    f"Rows shown: {len(favorites)} (all today/tomorrow rows, sorted by mismatch_score)"
)

# do this if you need to drop rows, for example, rookie pitcher or pitchers not declared
# favorites_view = favorites_view.drop(index=[635, 628])
favorites_view

Core filter (>= 2/3): home=3, away=5 | Rows shown: 10 (all today/tomorrow rows, sorted by mismatch_score)


,game_date,detailed_state,favored_team,_mismatch_side,core_matches,mismatch_score,home_team_name,away_team_name,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,home_offense_platoon,away_offense_platoon,home_pen_roll14_fip,away_pen_roll14_fip,home_pen_season_fip,away_pen_season_fip,home_probable_pitcher,away_probable_pitcher,game_pk,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_era,away_pen_season_era,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras
645,2026-04-16,In Progress,Texas Rangers,away_favored,2,11.284138,Athletics,Texas Rangers,-17.777778,2.130849,-2.844276,25.701166,99.704683,74.003517,4.219048,4.406452,4.047368,4.105076,Jacob Lopez,Jack Leiter,825023,133,140,3.0,5.0,682052,683004,L,R,94.856601,97.700877,-1.111111,16.666667,6.263636,4.132787,NaN,True,2026,2026,6.521739,20.512821,7.492857,4.830769,4.263158,2.741117,4.928571,3.266129,0.171679,0.301375,0.145,0.855,NaN,2026,0,2,False,True,False,False
639,2026-04-16,Final,Pittsburgh Pirates,home_favored,2,9.542385,Pittsburgh Pirates,Washington Nationals,8.184919,-2.533207,-1.990993,1.110036,100.674091,99.564056,3.414516,6.649618,4.244928,6.369841,Braxton Ashcraft,Foster Griffin,823396,134,120,7.0,8.0,677952,656492,R,L,104.527139,106.518132,21.978022,13.793103,1.776471,4.309677,0.0,True,2026,2026,39.024390,2.380952,0.281818,4.745161,3.913043,5.714286,3.919355,6.389313,-0.830411,0.279777,0.005,0.995,NaN,2026,2,0,True,False,True,False
647,2026-04-16,Pre-Game,Houston Astros,home_favored,2,7.135624,Houston Astros,Colorado Rockies,13.368726,0.654545,16.781228,16.172128,111.517367,95.345240,6.141379,4.268831,6.044954,4.337668,Ryan Weiss,Juan Mejia,824208,117,115,0.0,0.0,680802,675848,R,R,113.628822,96.847594,16.071429,2.702703,5.554545,4.900000,NaN,True,2026,2026,13.953488,4.545455,5.350000,5.900000,6.068807,3.753363,6.889655,3.331169,0.096425,-0.068837,0.605,0.395,NaN,2026,2,1,True,False,False,False
643,2026-04-16,Final,Toronto Blue Jays,away_favored,2,6.473566,Milwaukee Brewers,Toronto Blue Jays,-10.316414,1.048682,5.404124,-7.572725,87.631887,95.204613,3.785714,3.502985,3.248352,3.376699,Brandon Sproat,Patrick Corbin,823801,158,141,2.0,1.0,687075,571578,R,L,102.251718,96.847594,6.756757,17.073171,6.217647,5.168966,1.0,True,2026,2026,3.125000,9.523810,5.418182,10.350000,4.153846,4.063107,6.428571,3.425373,0.537363,0.126286,0.995,0.005,NaN,2026,0,2,False,True,False,True
644,2026-04-16,Final,Tampa Bay Rays,away_favored,3,6.388043,Chicago White Sox,Tampa Bay Rays,-5.147059,1.701109,-18.914435,-7.556940,98.475967,106.032907,3.471429,5.369565,4.521965,5.737931,Jordan Leasure,Steven Matz,824614,145,139,3.0,5.0,673929,571927,R,L,85.612704,104.527139,12.500000,17.647059,5.035484,3.334375,0.0,True,2026,2026,13.636364,26.190476,4.864706,1.736364,5.618497,6.982759,3.342857,5.869565,-1.050537,-0.368366,0.005,0.995,NaN,2026,0,3,False,True,False,False
642,2026-04-16,Final,New York Yankees,home_favored,1,6.087821,New York Yankees,Los Angeles Angels,-3.443214,-1.023333,-6.257407,-34.290739,78.399766,112.690504,3.250000,5.537500,2.831844,4.682915,Max Fried,Brent Suter,823560,147,108,4.0,11.0,608331,608718,L,L,98.838587,105.095994,11.811024,15.254237,3.010000,4.033333,0.0,True,2026,2026,12.280702,11.764706,2.350000,3.988889,3.469274,4.613065,4.275000,5.906250,0.418156,0.854585,0.005,0.995,NaN,2026,1,1,False,False,False,False
641,2026-04-16,In Progress,Detroit Tigers,home_favored,1,5.859141,Detroit Tigers,Kansas City Royals,2.657393,-1.645

In [10]:
import numpy as np
import pandas as pd

s = favorites_view.copy()

# =========================================================
# V6 MODEL — SLATE-NORMALIZED EDGE SYSTEM
# =========================================================

# =========================================================
# 1. NORMALIZED SIGNALS (STABLE SCALING)
# =========================================================

SP_KBB_ABS = 2.5
SP_XFIP_ABS = 0.75
OFFENSE_ABS = 5.0
PLATOON_ABS = 12.0
PEN_ABS = 1.5

kbb_norm = s["sp_kbb_diff"] / SP_KBB_ABS
xfip_norm = -s["sp_xfip_diff"] / SP_XFIP_ABS
off_norm = s["offense_diff"] / OFFENSE_ABS
platoon_norm = s["offense_platoon_diff"].fillna(0) / PLATOON_ABS

pen_gap = s["away_pen_roll14_fip"] - s["home_pen_roll14_fip"]
pen_norm = pen_gap / PEN_ABS

# =========================================================
# 2. STRUCTURED SIGNAL COMPONENTS
# =========================================================

sp_vector = (0.80 * kbb_norm) + (0.20 * xfip_norm)
off_vector = off_norm + (0.15 * platoon_norm)
pen_vector = 0.5 * pen_norm

signal_matrix = np.vstack([sp_vector, off_vector, pen_vector])

sp_vector = np.clip(sp_vector, -2, 2)
off_vector = np.clip(off_vector, -2, 2)
pen_vector = np.clip(pen_vector, -2, 2)

signal_matrix = np.vstack([sp_vector, off_vector, pen_vector])

# =========================================================
# 3. EDGE CONSTRUCTION
# =========================================================

sign_matrix = np.sign(signal_matrix)
mag_matrix = np.abs(signal_matrix)

mean_sign = np.mean(sign_matrix, axis=0)
mean_mag = np.mean(mag_matrix, axis=0)

raw_edge = mean_sign * mean_mag

# =========================================================
# 4. CONFIDENCE STRUCTURE
# =========================================================

agreement = 1 - np.mean(np.var(sign_matrix, axis=0))
direction_purity = np.abs(mean_sign)
mag_consistency = 1 / (1 + np.std(signal_matrix, axis=0))

coherence_score = (
    0.45 * agreement +
    0.30 * direction_purity +
    0.25 * mag_consistency
)

instability = np.std(signal_matrix, axis=0)
instability_penalty = 1 / (1 + instability)

risk_penalty = np.tanh(
    0.35 * (np.sign(sp_vector) != np.sign(off_vector)).astype(int) +
    0.45 * instability +
    0.20 * np.abs(sp_vector - off_vector)
)

trap_score = np.abs(raw_edge) * (1 - coherence_score)

quality_score = (
    raw_edge *
    coherence_score *
    instability_penalty *
    (1 / (1 + risk_penalty))
)

risk_adjusted_edge = quality_score * np.exp(-trap_score)

# =========================================================
# 5. SLATE NORMALIZATION
# =========================================================

s["edge_rank_pct"] = risk_adjusted_edge.rank(pct=True)
s["edge_z"] = (risk_adjusted_edge - risk_adjusted_edge.mean()) / (risk_adjusted_edge.std() + 1e-9)

# =========================================================
# 6. DECISION LAYER (SLATE RELATIVE)
# =========================================================

s["decision"] = np.select(
    [
        s["edge_rank_pct"] >= 0.90,
        s["edge_rank_pct"] >= 0.75
    ],
    ["BET", "LEAN"],
    default="NO BET"
)

# =========================================================
# 7. TIERS
# =========================================================

s["tier"] = np.select(
    [
        (s["decision"] == "BET") & (s["edge_rank_pct"] >= 0.97),
        (s["decision"] == "BET")
    ],
    ["A (Strong Bet)", "B (Playable Bet)"],
    default=np.where(
        s["decision"] == "LEAN",
        "C (Lean)",
        "D (Pass)"
    )
)

# =========================================================
# 8. TEAM LOGIC
# =========================================================

prefer_home = sp_vector >= 0

s["favored_team"] = np.where(
    prefer_home,
    s["home_team_name"],
    s["away_team_name"]
)

s["_mismatch_side"] = np.where(
    prefer_home,
    "home_favored",
    "away_favored"
)

# =========================================================
# 9. INTERPRETABILITY
# =========================================================

s["signal_agreement"] = np.sum(sign_matrix > 0, axis=0)
s["signal_conflict"] = np.sum(sign_matrix < 0, axis=0)
s["direction_conflict"] = (np.sum(sign_matrix < 0, axis=0) > 0).astype(int)
s["instability"] = instability

s["coherence_score"] = coherence_score
s["quality_score"] = quality_score
s["risk_adjusted_edge"] = risk_adjusted_edge
s["trap_score"] = trap_score

# =========================================================
# 10. FEATURES
# =========================================================

s["core_matches"] = (
    (np.abs(kbb_norm) > 1).astype(int) +
    (np.abs(xfip_norm) > 1).astype(int) +
    (np.abs(off_norm) > 1).astype(int)
)

# =========================================================
# 11. MARKET
# =========================================================

if "kalshi_home_implied" in s.columns:
    s["implied_prob"] = np.where(
        prefer_home,
        s["kalshi_home_implied"],
        s["kalshi_away_implied"]
    )
    s["market_edge"] = risk_adjusted_edge - s["implied_prob"]

# =========================================================
# 12. SORT
# =========================================================

s = s.sort_values(["risk_adjusted_edge", "coherence_score"], ascending=[False, False])

# =========================================================
# 13. OUTPUT
# =========================================================

priority_cols = [
    "game_date",
    "home_team_name",
    "away_team_name",
    "favored_team",
    "_mismatch_side",
    "risk_adjusted_edge",
    "edge_rank_pct",
    "edge_z",
    "decision",
    "tier",
    "coherence_score",
    "trap_score",
    "instability",
    "signal_agreement",
    "signal_conflict",
    "direction_conflict",
    "core_matches",
    "home_probable_pitcher",
    "away_probable_pitcher",
    "sp_kbb_diff",
    "sp_xfip_diff",
    "offense_diff",
    "offense_platoon_diff",
    "home_pen_roll14_fip",
    "away_pen_roll14_fip"
]

priority_cols = [c for c in priority_cols if c in s.columns]
rest_cols = [c for c in s.columns if c not in priority_cols]

final_view = s[priority_cols + rest_cols]

print(f"V6 complete — games evaluated: {len(final_view)}")
final_view


V6 complete — games evaluated: 10


,game_date,home_team_name,away_team_name,favored_team,_mismatch_side,risk_adjusted_edge,edge_rank_pct,edge_z,decision,tier,coherence_score,trap_score,instability,signal_agreement,signal_conflict,direction_conflict,core_matches,home_probable_pitcher,away_probable_pitcher,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,home_pen_roll14_fip,away_pen_roll14_fip,detailed_state,mismatch_score,home_offense_platoon,away_offense_platoon,home_pen_season_fip,away_pen_season_fip,game_pk,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_era,away_pen_season_era,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras,quality_score,implied_prob,market_edge
641,2026-04-16,Detroit Tigers,Kansas City Royals,Detroit Tigers,home_favored,0.197954,1.0,2.674032,BET,A (Strong Bet),0.556603,0.510970,0.500576,3,0,0,3,Keider Montero,Kris Bubic,2.657393,-1.645858,8.675041,-3.962414,3.652632,5.100000,In Progress,5.859141,89.976553,93.938968,4.227273,4.868212,824289,116,118,7.0,8.0,672456,663460,R,L,100.687367,92.012325,20.634921,17.977528,1.630612,3.276471,NaN,True,2026,2026,21.052632,32.608696,1.745161,2.016667,2.945455,4.291391,1.657895,4.736842,-0.574641,0.231788,0.305,0.695,NaN,2026,1,0,False,False,False,False,0.329972,0.305,-0.107046
647,2026-04-16,Houston Astros,Colorado Rockies,Houston Astros,home_favored,0.032157,0.9,0.269603,BET,B (Playable Bet),0.301754,0.358757,1.237052,2,1,1,2,Ryan Weiss,Juan Mejia,13.368726,0.654545,16.781228,16.172128,6.141379,4.268831,Pre-Game,7.135624,111.517367,95.345240,6.044954,4.337668,824208,117,115,0.0,0.0,680802,675848,R,R,113.628822,96.847594,16.071429,2.702703,5.554545,4.900000,NaN,True,2026,2026,13.953488,4.545455,5.350000,5.900000,6.068807,3.753363,6.889655,3.331169,0.096425,-0.068837,0.605,0.395,NaN,2026,2,1,True,False,False,False,0.046035,0.605,-0.572843
639,2026-04-16,Pittsburgh Pirates,Washington Nationals,Pittsburgh Pirates,home_favored,0.025456,0.8,0.172415,LEAN,C (Lean),0.316153,0.263105,0.981714,2,1,1,2,Braxton Ashcraft,Foster Griffin,8.184919,-2.533207,-1.990993,1.110036,3.414516,6.649618,Final,9.542385,100.674091,99.564056,4.244928,6.369841,823396,134,120,7.0,8.0,677952,656492,R,L,104.527139,106.518132,21.978022,13.793103,1.776471,4.309677,0.0,True,2026,2026,39.024390,2.380952,0.281818,4.745161,3.913043,5.714286,3.919355,6.389313,-0.830411,0.279777,0.005,0.995,NaN,2026,2,0,True,False,True,False,0.033117,0.005,0.020456
648,2026-04-16,San Diego Padres,Seattle Mariners,Seattle Mariners,away_favored,0.019257,0.7,0.082525,NO BET,D (Pass),0.301480,0.243474,1.242548,2,1,1,2,Walker Buehler,Luis Castillo,-4.808278,0.710526,6.399621,0.843763,2.865957,3.221212,Pre-Game,3.527520,101.532836,100.689073,2.686667,3.259763,823313,135,136,0.0,0.0,621111,622491,R,R,100.118511,93.718891,11.320755,16.129032,3.810526,3.100000,NaN,True,2026,2026,14.705882,12.195122,2.292308,4.242857,3.120000,2.875740,2.872340,3.000000,0.179291,-0.038551,0.485,0.515,NaN,2026,0,2,False,True,False,False,0.024566,0.515,-0.495743
640,2026-04-16,Cincinnati Reds,San Francisco Giants,San Francisco Giants,away_favored,-0.017598,0.6,-0.451956,NO BET,D (Pass),0.324621,0.141095,0.857068,1,2,1,2,Chase Burns,Landen Roupp,-4.116162,1.911545,-0.142214,3.656307,4.100000,3.991892,Final,5.387787,87.188862,83.532555,4.176555,4.334286,824530,113,137,0.0,3.0,695505,694738,R,R,89.879118,90.021332,14.772727,18.888889,4.085075,2.173529,0.0,True,2026,2026,11.764706,16.666667,5.923529,1.881250,2.712919,4.165714,2.045455,4.62

In [11]:
def decision_layer(df):

    conditions = [
        # BET (strong + clean)
        (
            (df["risk_adjusted_edge"] > 1.0) &
            (df["coherence_score"] > 0.60) &
            (df["instability"] < 1.2)
        ),

        # LEAN (some edge but imperfect structure)
        (
            (df["risk_adjusted_edge"] > 0.5) &
            (df["coherence_score"] > 0.45)
        ),

        # NO BET (everything else)
    ]

    choices = ["BET", "LEAN"]

    df["decision"] = np.select(conditions, choices, default="NO BET")

    return df

decision_layer(final_view)

,game_date,home_team_name,away_team_name,favored_team,_mismatch_side,risk_adjusted_edge,edge_rank_pct,edge_z,decision,tier,coherence_score,trap_score,instability,signal_agreement,signal_conflict,direction_conflict,core_matches,home_probable_pitcher,away_probable_pitcher,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,home_pen_roll14_fip,away_pen_roll14_fip,detailed_state,mismatch_score,home_offense_platoon,away_offense_platoon,home_pen_season_fip,away_pen_season_fip,game_pk,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_era,away_pen_season_era,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras,quality_score,implied_prob,market_edge
641,2026-04-16,Detroit Tigers,Kansas City Royals,Detroit Tigers,home_favored,0.197954,1.0,2.674032,NO BET,A (Strong Bet),0.556603,0.510970,0.500576,3,0,0,3,Keider Montero,Kris Bubic,2.657393,-1.645858,8.675041,-3.962414,3.652632,5.100000,In Progress,5.859141,89.976553,93.938968,4.227273,4.868212,824289,116,118,7.0,8.0,672456,663460,R,L,100.687367,92.012325,20.634921,17.977528,1.630612,3.276471,NaN,True,2026,2026,21.052632,32.608696,1.745161,2.016667,2.945455,4.291391,1.657895,4.736842,-0.574641,0.231788,0.305,0.695,NaN,2026,1,0,False,False,False,False,0.329972,0.305,-0.107046
647,2026-04-16,Houston Astros,Colorado Rockies,Houston Astros,home_favored,0.032157,0.9,0.269603,NO BET,B (Playable Bet),0.301754,0.358757,1.237052,2,1,1,2,Ryan Weiss,Juan Mejia,13.368726,0.654545,16.781228,16.172128,6.141379,4.268831,Pre-Game,7.135624,111.517367,95.345240,6.044954,4.337668,824208,117,115,0.0,0.0,680802,675848,R,R,113.628822,96.847594,16.071429,2.702703,5.554545,4.900000,NaN,True,2026,2026,13.953488,4.545455,5.350000,5.900000,6.068807,3.753363,6.889655,3.331169,0.096425,-0.068837,0.605,0.395,NaN,2026,2,1,True,False,False,False,0.046035,0.605,-0.572843
639,2026-04-16,Pittsburgh Pirates,Washington Nationals,Pittsburgh Pirates,home_favored,0.025456,0.8,0.172415,NO BET,C (Lean),0.316153,0.263105,0.981714,2,1,1,2,Braxton Ashcraft,Foster Griffin,8.184919,-2.533207,-1.990993,1.110036,3.414516,6.649618,Final,9.542385,100.674091,99.564056,4.244928,6.369841,823396,134,120,7.0,8.0,677952,656492,R,L,104.527139,106.518132,21.978022,13.793103,1.776471,4.309677,0.0,True,2026,2026,39.024390,2.380952,0.281818,4.745161,3.913043,5.714286,3.919355,6.389313,-0.830411,0.279777,0.005,0.995,NaN,2026,2,0,True,False,True,False,0.033117,0.005,0.020456
648,2026-04-16,San Diego Padres,Seattle Mariners,Seattle Mariners,away_favored,0.019257,0.7,0.082525,NO BET,D (Pass),0.301480,0.243474,1.242548,2,1,1,2,Walker Buehler,Luis Castillo,-4.808278,0.710526,6.399621,0.843763,2.865957,3.221212,Pre-Game,3.527520,101.532836,100.689073,2.686667,3.259763,823313,135,136,0.0,0.0,621111,622491,R,R,100.118511,93.718891,11.320755,16.129032,3.810526,3.100000,NaN,True,2026,2026,14.705882,12.195122,2.292308,4.242857,3.120000,2.875740,2.872340,3.000000,0.179291,-0.038551,0.485,0.515,NaN,2026,0,2,False,True,False,False,0.024566,0.515,-0.495743
640,2026-04-16,Cincinnati Reds,San Francisco Giants,San Francisco Giants,away_favored,-0.017598,0.6,-0.451956,NO BET,D (Pass),0.324621,0.141095,0.857068,1,2,1,2,Chase Burns,Landen Roupp,-4.116162,1.911545,-0.142214,3.656307,4.100000,3.991892,Final,5.387787,87.188862,83.532555,4.176555,4.334286,824530,113,137,0.0,3.0,695505,694738,R,R,89.879118,90.021332,14.772727,18.888889,4.085075,2.173529,0.0,True,2026,2026,11.764706,16.666667,5.923529,1.881250,2.712919,4.165714,2.045

In [12]:
def confidence_tier(df):

    df["tier"] = np.where(
        (df["decision"] == "BET") & (df["coherence_score"] > 0.75),
        "A (Strong Bet)",
        np.where(
            (df["decision"] == "BET"),
            "B (Playable Bet)",
            np.where(
                (df["decision"] == "LEAN"),
                "C (Lean Only)",
                "D (Pass)"
            )
        )
    )

    return df

confidence_tier(final_view)

,game_date,home_team_name,away_team_name,favored_team,_mismatch_side,risk_adjusted_edge,edge_rank_pct,edge_z,decision,tier,coherence_score,trap_score,instability,signal_agreement,signal_conflict,direction_conflict,core_matches,home_probable_pitcher,away_probable_pitcher,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,home_pen_roll14_fip,away_pen_roll14_fip,detailed_state,mismatch_score,home_offense_platoon,away_offense_platoon,home_pen_season_fip,away_pen_season_fip,game_pk,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_era,away_pen_season_era,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras,quality_score,implied_prob,market_edge
641,2026-04-16,Detroit Tigers,Kansas City Royals,Detroit Tigers,home_favored,0.197954,1.0,2.674032,NO BET,D (Pass),0.556603,0.510970,0.500576,3,0,0,3,Keider Montero,Kris Bubic,2.657393,-1.645858,8.675041,-3.962414,3.652632,5.100000,In Progress,5.859141,89.976553,93.938968,4.227273,4.868212,824289,116,118,7.0,8.0,672456,663460,R,L,100.687367,92.012325,20.634921,17.977528,1.630612,3.276471,NaN,True,2026,2026,21.052632,32.608696,1.745161,2.016667,2.945455,4.291391,1.657895,4.736842,-0.574641,0.231788,0.305,0.695,NaN,2026,1,0,False,False,False,False,0.329972,0.305,-0.107046
647,2026-04-16,Houston Astros,Colorado Rockies,Houston Astros,home_favored,0.032157,0.9,0.269603,NO BET,D (Pass),0.301754,0.358757,1.237052,2,1,1,2,Ryan Weiss,Juan Mejia,13.368726,0.654545,16.781228,16.172128,6.141379,4.268831,Pre-Game,7.135624,111.517367,95.345240,6.044954,4.337668,824208,117,115,0.0,0.0,680802,675848,R,R,113.628822,96.847594,16.071429,2.702703,5.554545,4.900000,NaN,True,2026,2026,13.953488,4.545455,5.350000,5.900000,6.068807,3.753363,6.889655,3.331169,0.096425,-0.068837,0.605,0.395,NaN,2026,2,1,True,False,False,False,0.046035,0.605,-0.572843
639,2026-04-16,Pittsburgh Pirates,Washington Nationals,Pittsburgh Pirates,home_favored,0.025456,0.8,0.172415,NO BET,D (Pass),0.316153,0.263105,0.981714,2,1,1,2,Braxton Ashcraft,Foster Griffin,8.184919,-2.533207,-1.990993,1.110036,3.414516,6.649618,Final,9.542385,100.674091,99.564056,4.244928,6.369841,823396,134,120,7.0,8.0,677952,656492,R,L,104.527139,106.518132,21.978022,13.793103,1.776471,4.309677,0.0,True,2026,2026,39.024390,2.380952,0.281818,4.745161,3.913043,5.714286,3.919355,6.389313,-0.830411,0.279777,0.005,0.995,NaN,2026,2,0,True,False,True,False,0.033117,0.005,0.020456
648,2026-04-16,San Diego Padres,Seattle Mariners,Seattle Mariners,away_favored,0.019257,0.7,0.082525,NO BET,D (Pass),0.301480,0.243474,1.242548,2,1,1,2,Walker Buehler,Luis Castillo,-4.808278,0.710526,6.399621,0.843763,2.865957,3.221212,Pre-Game,3.527520,101.532836,100.689073,2.686667,3.259763,823313,135,136,0.0,0.0,621111,622491,R,R,100.118511,93.718891,11.320755,16.129032,3.810526,3.100000,NaN,True,2026,2026,14.705882,12.195122,2.292308,4.242857,3.120000,2.875740,2.872340,3.000000,0.179291,-0.038551,0.485,0.515,NaN,2026,0,2,False,True,False,False,0.024566,0.515,-0.495743
640,2026-04-16,Cincinnati Reds,San Francisco Giants,San Francisco Giants,away_favored,-0.017598,0.6,-0.451956,NO BET,D (Pass),0.324621,0.141095,0.857068,1,2,1,2,Chase Burns,Landen Roupp,-4.116162,1.911545,-0.142214,3.656307,4.100000,3.991892,Final,5.387787,87.188862,83.532555,4.176555,4.334286,824530,113,137,0.0,3.0,695505,694738,R,R,89.879118,90.021332,14.772727,18.888889,4.085075,2.173529,0.0,True,2026,2026,11.764706,16.666667,5.923529,1.881250,2.712919,4.165714,2.045455,4.621622,-